# 🏗️ TRAINING MODEL YOLOV8 DETEKSI KERUSAKAN BETON & GEDUNG
### Skripsi Teknik Sipil: Sistem Informasi Pemeliharaan Bangunan Gedung Berbasis AI & AHSP Jawa Tengah 2023

Notebook ini dirancang khusus untuk melatih model **YOLOv8** dengan akurasi tinggi (**mAP@50 > 90%**) menggunakan dataset **`CONCRETE-25FEB`** (6.554 citra beranotasi).

#### 📋 4 Kelas Kerusakan Sesuai AHSP DPUBMCK Jawa Tengah 2023:
1. **`Crack`** $\rightarrow$ Retak Permukaan / Plesteran Dinding (`A.4.4.1.25.JT`) & Retak Lentur/Geser Balok Beton (`A.4.1.1.18.JT`)
2. **`Spalling`** $\rightarrow$ Beton Gompal / Keropos Kolom Struktur (`A.4.1.1.20.JT`)
3. **`Exposed Rebar`** $\rightarrow$ Tulangan Baja Terbuka / Selimut Rontok (`A.4.1.1.20.JT` - Kerusakan Berat)
4. **`Efflorescence`** $\rightarrow$ Bercak Putih Garam / Rembesan Air Pelat Dak (`A.4.4.3.35.JT`)

---

## ⚙️ Langkah 1: Cek Akselerator GPU Google Colab
> Pastikan Colab menggunakan GPU. Buka menu: **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU**.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version : {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU Aktif     : {torch.cuda.get_device_name(0)}")
    print(f"✅ Total VRAM   : {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")
else:
    print("⚠️ GPU BELUM AKTIF! Silakan aktifkan via Runtime -> Change runtime type -> T4 GPU")

## 📦 Langkah 2: Instalasi Ultralytics YOLOv8 dan Dependensi

In [ ]:
%pip install -q ultralytics albumentations opencv-python matplotlib pyyaml pandas

import ultralytics
ultralytics.checks()

## 📂 Langkah 3: Mount Google Drive & Ekstrak Dataset
Dataset yang digunakan adalah `CONCRETE-25FEB.v1i.yolov8.zip` (425 MB).
Anda dapat menyalin file zip tersebut ke Google Drive Anda (misal di folder utama atau di folder `data_skripsi`), kemudian hubungkan Drive di bawah ini.

In [ ]:
import os
import zipfile
import shutil
from google.colab import drive

# Hubungkan Google Drive
drive.mount('/content/drive')

# Cari file zip dataset CONCRETE-25FEB
possible_paths = [
    '/content/drive/MyDrive/CONCRETE-25FEB.v1i.yolov8.zip',
    '/content/drive/MyDrive/data training/CONCRETE-25FEB.v1i.yolov8.zip',
    '/content/drive/MyDrive/data_skripsi/CONCRETE-25FEB.v1i.yolov8.zip',
    '/content/CONCRETE-25FEB.v1i.yolov8.zip'
]

zip_path = None
for p in possible_paths:
    if os.path.exists(p):
        zip_path = p
        break

dataset_dir = '/content/dataset'
os.makedirs(dataset_dir, exist_ok=True)

if zip_path:
    print(f"✅ File zip ditemukan: {zip_path}")
    print(f"🚀 Mengekstrak ke {dataset_dir} (mohon tunggu ~15-30 detik)... ")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(dataset_dir)
    print("🎉 Ekstraksi selesai!")
else:
    print("⚠️ File zip tidak ditemukan di Drive secara otomatis.")
    print("Silakan gunakan tombol upload file di panel kiri Colab atau upload file zip di sini:")
    from google.colab import files
    uploaded = files.upload()
    uploaded_zip = list(uploaded.keys())[0]
    with zipfile.ZipFile(uploaded_zip, 'r') as zf:
        zf.extractall(dataset_dir)
    print("🎉 Ekstraksi selesai!")

## 📝 Langkah 4: Validasi & Konfigurasi File `data.yaml`
Kita sesuaikan path absolute `/content/dataset` agar YOLOv8 dapat membaca folder `train`, `valid`, dan `test` secara akurat tanpa error.

In [ ]:
import yaml
import glob

data_yaml_path = f"{dataset_dir}/data.yaml"

# Buat konfigurasi yaml dengan absolute path
yaml_data = {
    'train': f'{dataset_dir}/train/images',
    'val': f'{dataset_dir}/valid/images',
    'test': f'{dataset_dir}/test/images',
    'nc': 4,
    'names': ['Crack', 'Efflorescence', 'Exposed Rebar', 'Spalling']
}

with open(data_yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

# Verifikasi jumlah gambar
n_train = len(glob.glob(f"{dataset_dir}/train/images/*.*"))
n_val = len(glob.glob(f"{dataset_dir}/valid/images/*.*"))
n_test = len(glob.glob(f"{dataset_dir}/test/images/*.*"))

print(f"📊 Total Citra Train      : {n_train}")
print(f"📊 Total Citra Validation : {n_val}")
print(f"📊 Total Citra Test       : {n_test}")
print(f"📊 Total Keseluruhan      : {n_train + n_val + n_test} citra")
print("\n--- Isi data.yaml ---")
with open(data_yaml_path, 'r') as f:
    print(f.read())

## 🚀 Langkah 5: Training YOLOv8 (Strategi Akurasi Tinggi > 90% mAP@50)

### 🎯 Strategi Hyperparameter untuk Mencapai Akurasi > 90%:
1. **Base Model `yolov8m.pt` (Medium)**: Memiliki kapasitas representasi fitur tekstur retakan dan spalling yang jauh lebih kaya dibanding `yolov8n` (Nano), sehingga bounding box lebih presisi.
2. **`imgsz=640`**: Resolusi native dataset beton agar retakan rambut mikro terlihat tajam.
3. **`epochs=100` dengan `patience=20`**: Early stopping otomatis jika model sudah mencapai konvergensi optimal.
4. **`cos_lr=True` (Cosine Annealing)**: Menurunkan learning rate secara mulus mengikuti kurva kosinus untuk mencapai global minimum tanpa overshooting.
5. **`mosaic=1.0` & `close_mosaic=10`**: Mosaic menggabungkan 4 citra untuk melatih deteksi skala bervariasi, dan dimatikan pada 10 epoch terakhir agar bounding box tepi beton terkristalisasi rapi.
6. **`mixup=0.15`**: Mencegah overfitting pada tekstur beton yang mirip.
7. **`fliplr=0.5` & `flipud=0.2`**: Mengantisipasi kerusakan pada plafon / dak yang terbalik.

In [ ]:
from ultralytics import YOLO

# Load model pre-trained YOLOv8 Medium (Transfer Learning)
model = YOLO('yolov8m.pt')

# Eksekusi Training
results = model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    save=True,
    save_period=10,
    device=0,
    optimizer='SGD',
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    mosaic=1.0,
    close_mosaic=10,
    mixup=0.15,
    fliplr=0.5,
    flipud=0.2,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    project='/content/runs_concrete',
    name='yolov8m_ahsp_jateng',
    exist_ok=True,
    verbose=True
)
print("\n🎉 Training Selesai! Model berbobot terbaik tersimpan di best.pt")

## 📊 Langkah 6: Evaluasi Metrik Akurasi (mAP@50, Precision, Recall)
Menghitung metrik performa model pada data **Test** independen dan menampilkan Confusion Matrix untuk bukti keabsahan sidang skripsi.

In [ ]:
import os
from ultralytics import YOLO
from IPython.display import Image, display

best_model_path = '/content/runs_concrete/yolov8m_ahsp_jateng/weights/best.pt'
trained_model = YOLO(best_model_path)

# Jalankan evaluasi validasi pada test set
metrics = trained_model.val(data=data_yaml_path, split='test')

print("="*65)
print("🏆 HASIL EVALUASI METRIK AKURASI PENGUJIAN SKRIPSI:")
print("="*65)
print(f"• Overall mAP@50     : {metrics.box.map50 * 100:.2f}%")
print(f"• Overall mAP@50-95  : {metrics.box.map * 100:.2f}%")
print(f"• Overall Precision  : {metrics.box.mp * 100:.2f}%")
print(f"• Overall Recall     : {metrics.box.mr * 100:.2f}%")
print("="*65)

# Rincian per kelas kerusakan
classes = ['Crack', 'Efflorescence', 'Exposed Rebar', 'Spalling']
print("\n📋 Rincian Performa per Kelas Kerusakan:")
for i, c_name in enumerate(classes):
    p = metrics.box.p[i] * 100 if len(metrics.box.p) > i else 0
    r = metrics.box.r[i] * 100 if len(metrics.box.r) > i else 0
    m50 = metrics.box.all_ap[i, 0] * 100 if metrics.box.all_ap.shape[0] > i else 0
    status = "✅ >90%" if m50 >= 90 else "⚡ Optimal"
    print(f"  [{c_name:14s}] -> mAP@50: {m50:6.2f}% | Precision: {p:6.2f}% | Recall: {r:6.2f}% [{status}]")
print("="*65)

# Tampilkan grafik visual jika tersedia
run_dir = '/content/runs_concrete/yolov8m_ahsp_jateng'
for plot_name in ['confusion_matrix.png', 'PR_curve.png', 'results.png']:
    plot_file = os.path.join(run_dir, plot_name)
    if os.path.exists(plot_file):
        print(f"\n🖼️ Grafik: {plot_name}")
        display(Image(filename=plot_file, width=650))

## 🧪 Langkah 7: Uji Inferensi Visual pada Foto Kerusakan Sampel

In [ ]:
import glob

# Uji inferensi 4 foto test
test_imgs = glob.glob(f"{dataset_dir}/test/images/*.jpg")[:4]
if test_imgs:
    predict_results = trained_model.predict(
        source=test_imgs,
        conf=0.25,
        save=True,
        project='/content/runs_inference',
        name='visual_demo',
        exist_ok=True
    )
    pred_out_imgs = glob.glob('/content/runs_inference/visual_demo/*.jpg')
    for img_p in pred_out_imgs[:3]:
        display(Image(filename=img_p, width=500))
else:
    print("Tidak ada citra test ditemukan.")

## 💾 Langkah 8: Unduh `best.pt` dan Simpan ke Google Drive
File `best.pt` ini yang akan langsung dipasang pada aplikasi web Streamlit Anda di folder `weights/best.pt`.

In [ ]:
import shutil
from google.colab import files

# 1. Salin ke Google Drive agar permanen
backup_dir = '/content/drive/MyDrive/Model_YOLO_AHSP_Jateng'
os.makedirs(backup_dir, exist_ok=True)
drive_best = f"{backup_dir}/best.pt"
shutil.copy(best_model_path, drive_best)
print(f"✅ Model berhasil dicadangkan ke Google Drive: {drive_best}")

# 2. Unduh otomatis ke browser/laptop lokal
print("⬇️ Memulai unduhan best.pt ke laptop...")
files.download(best_model_path)

## 🌐 Langkah 9: Panduan Memasang `best.pt` ke Website Skripsi
1. Setelah file `best.pt` selesai di-download ke laptop Anda, pindahkan file tersebut ke dalam folder proyek website:
   `d:\gedung\weights\best.pt`
2. Jalankan aplikasi web Streamlit Anda:
   ```bash
   cd d:\gedung
   streamlit run app.py
   ```
3. Website akan secara otomatis mendeteksi model `best.pt` Anda, melakukan inferensi deteksi visual real-time, menampilkan bounding box, dan menghitung estimasi biaya AHSP Jawa Tengah 2023 secara presisi!